#  Student Productivity Prediction — Linear Regression

**Dataset:** `ultimate_student_productivity_dataset_5000.csv`  
**Task:** Regression — Prediksi **Productivity Score**  
**Model:** Ridge Regression

---
###  Daftar Isi
1. Cara Melihat Tipe Data
2. Dataset Bisa Digunakan Untuk Apa
3. Model Yang Bisa Digunakan
4. Parameter Yang Bisa Diubah/Disetel
5. Evaluasi Yang Dipakai
6. Cara Mengetahui Evaluasi Bagus atau Tidak
7. Cara Mengoptimasi Model
8. Cara Menyimpan Model
9. Cara Menggunakan Model Hasil Training

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, os, warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression, Ridge, Lasso, RidgeCV, ElasticNet
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print('Libraries loaded ')

---
## 1.  Cara Melihat Tipe Data

In [ ]:
df = pd.read_csv('../ultimate_student_productivity_dataset_5000.csv')
print(f'Shape: {df.shape}')
df.info()

In [ ]:
print('Statistik kolom numerik:')
df.describe().round(2)

In [ ]:
print('Kolom kategorik:')
for col in df.select_dtypes(include='object').columns:
    print(f'  {col}: {df[col].unique()}')

print(f'\nMissing values: {df.isnull().sum().sum()}')

# Distribusi target
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['productivity_score'].hist(bins=30, ax=axes[0], color='steelblue')
axes[0].set_title('Distribusi Productivity Score')
df['exam_score'].hist(bins=30, ax=axes[1], color='green')
axes[1].set_title('Distribusi Exam Score')
plt.tight_layout(); plt.show()

---
## 2.  Dataset Bisa Digunakan Untuk Apa

| Tujuan | Target | Jenis |
|--------|--------|-------|
| **Prediksi skor produktivitas** ← (ini) | `productivity_score` | Regression |
| Prediksi nilai ujian | `exam_score` | Regression |
| Klasifikasi produktivitas tinggi/rendah | binary dari median | Classification |
| Pengelompokan tipe mahasiswa | — | Clustering |
| Analisis burnout | `burnout_level` | Regression/Classification |

**Kolom:** `student_id, age, gender, academic_level, study_hours, self_study_hours, online_classes_hours, social_media_hours, gaming_hours, sleep_hours, screen_time_hours, exercise_minutes, caffeine_intake_mg, part_time_job, upcoming_deadline, internet_quality, mental_health_score, focus_index, burnout_level, productivity_score, exam_score`

In [ ]:
# Heatmap korelasi
plt.figure(figsize=(14, 10))
num_df = df.select_dtypes(include=np.number).drop(columns=['student_id'])
sns.heatmap(num_df.corr(), annot=True, fmt='.2f', cmap='coolwarm', square=True, linewidths=0.5)
plt.title('Korelasi Antar Fitur')
plt.tight_layout(); plt.show()

In [ ]:
# Preprocessing
target = 'productivity_score'
drop_cols = ['student_id', 'exam_score']  # exam_score mungkin berkorelasi tinggi

df_proc = df.drop(columns=drop_cols).copy()
le = LabelEncoder()
for col in df_proc.select_dtypes(include='object').columns:
    df_proc[col] = le.fit_transform(df_proc[col].astype(str))

X = df_proc.drop(columns=[target])
y = df_proc[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

---
## 3.  Kenapa Linear Regression?

Linear Regression mencari hyperplane:
$$\hat{y} = w_0 + w_1 x_1 + w_2 x_2 + \ldots + w_n x_n$$

**Cocok untuk produktivitas karena:**
- Banyak faktor (study hours, sleep, mental health) mungkin berkontribusi secara **additive**
- Koefisien mudah diinterpretasikan: "setiap jam belajar tambahan meningkatkan produktivitas X poin"
- Cepat dan bisa digunakan sebagai baseline

**Ridge Regression** menambahkan regularisasi L2 untuk menghindari overfitting ketika fitur saling berkorelasi.

In [ ]:
models = {
    'LinearRegression': Pipeline([('s', StandardScaler()), ('m', LinearRegression())]),
    'Ridge(α=0.1)':     Pipeline([('s', StandardScaler()), ('m', Ridge(alpha=0.1))]),
    'Ridge(α=1.0)':     Pipeline([('s', StandardScaler()), ('m', Ridge(alpha=1.0))]),
    'Ridge(α=10)':      Pipeline([('s', StandardScaler()), ('m', Ridge(alpha=10.0))]),
    'Lasso(α=0.01)':    Pipeline([('s', StandardScaler()), ('m', Lasso(alpha=0.01))]),
    'ElasticNet':       Pipeline([('s', StandardScaler()), ('m', ElasticNet(alpha=0.1, l1_ratio=0.5))])
}

results = {}
for name, mdl in models.items():
    mdl.fit(X_train, y_train)
    yp = mdl.predict(X_test)
    results[name] = {
        'R²': r2_score(y_test, yp),
        'RMSE': np.sqrt(mean_squared_error(y_test, yp)),
        'MAE': mean_absolute_error(y_test, yp)
    }

res_df = pd.DataFrame(results).T.sort_values('R²', ascending=False)
print('Perbandingan model linear:')
print(res_df.to_string())

---
## 4.  Parameter Yang Bisa Diubah / Disetel

| Model | Parameter | Range | Penjelasan |
|-------|-----------|-------|------------|
| Ridge | `alpha` | 0.001–1000 | Kekuatan regularisasi L2 |
| Lasso | `alpha` | 0.001–100 | Kekuatan regularisasi L1 |
| ElasticNet | `alpha` | 0.001–100 | Total regularisasi |
| ElasticNet | `l1_ratio` | 0–1 | 0=pure Ridge, 1=pure Lasso |

**Cari alpha optimal dengan RidgeCV:**

In [ ]:
ridge_cv_model = Pipeline([
    ('scaler', StandardScaler()),
    ('ridge', RidgeCV(alphas=np.logspace(-3, 3, 50), cv=5))
])
ridge_cv_model.fit(X_train, y_train)
best_alpha = ridge_cv_model.named_steps['ridge'].alpha_
print(f'Best alpha: {best_alpha:.4f}')

yp_cv = ridge_cv_model.predict(X_test)
print(f'RidgeCV R²  : {r2_score(y_test, yp_cv):.4f}')
print(f'RidgeCV RMSE: {np.sqrt(mean_squared_error(y_test, yp_cv)):.4f}')

---
## 5.  Evaluasi Yang Dipakai

In [ ]:
y_pred = ridge_cv_model.predict(X_test)

mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)
mape = np.mean(np.abs((y_test - y_pred) / (y_test + 1e-8))) * 100

print('='*50)
print(' EVALUASI RIDGE REGRESSION (PRODUCTIVITY)')
print('='*50)
print(f'MAE  : {mae:.4f}')
print(f'RMSE : {rmse:.4f}')
print(f'R²   : {r2:.4f}')
print(f'MAPE : {mape:.2f}%')

# CV Score
cv_scores = cross_val_score(ridge_cv_model, X, y, cv=5, scoring='r2')
print(f'\nCross-Val R²: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_test, y_pred, alpha=0.4, color='steelblue')
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
axes[0].plot(lims, lims, 'r--', lw=2)
axes[0].set_xlabel('Aktual'); axes[0].set_ylabel('Prediksi')
axes[0].set_title(f'Actual vs Predicted (R²={r2:.3f})')

# Koefisien terbesar
coef = ridge_cv_model.named_steps['ridge'].coef_
coef_ser = pd.Series(coef, index=X.columns).sort_values(key=abs, ascending=False).head(10)
colors = ['tomato' if v < 0 else 'steelblue' for v in coef_ser.values]
coef_ser.plot(kind='barh', ax=axes[1], color=colors)
axes[1].set_title('Koefisien Terbesar')
axes[1].invert_yaxis()
axes[1].axvline(0, color='black')
plt.tight_layout(); plt.show()

---
## 6.  Cara Mengetahui Evaluasi Bagus atau Tidak

### Skala `productivity_score` perlu diketahui:

In [ ]:
score_range = y.max() - y.min()
print(f'Range productivity_score: {y.min():.1f} – {y.max():.1f} (range={score_range:.1f})')
print(f'Mean: {y.mean():.1f}, Std: {y.std():.1f}')
print(f'\nMAE  = {mae:.4f} ({mae/score_range*100:.1f}% dari range)')
print(f'RMSE = {rmse:.4f} ({rmse/score_range*100:.1f}% dari range)')
print(f'R²   = {r2:.4f}')

# Interpretasi
if r2 > 0.85:
    print('\n Excellent! Model menjelaskan >85% variansi')
elif r2 > 0.7:
    print('\n Good. Model cukup prediktif')
elif r2 > 0.5:
    print('\n Fair. Model bisa digunakan tapi ada ruang perbaikan')
else:
    print('\n Poor. Pertimbangkan model non-linear (Random Forest, XGBoost)')

---
## 7.  Cara Mengoptimasi Model

In [ ]:
# Coba Polynomial Features
from sklearn.preprocessing import PolynomialFeatures

poly_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('poly', PolynomialFeatures(degree=2, include_bias=False, interaction_only=True)),
    ('ridge', RidgeCV(alphas=np.logspace(-3, 3, 30), cv=5))
])
poly_pipe.fit(X_train, y_train)
yp_poly = poly_pipe.predict(X_test)
print(f'Polynomial (degree=2) R² : {r2_score(y_test, yp_poly):.4f}')
print(f'Polynomial RMSE          : {np.sqrt(mean_squared_error(y_test, yp_poly)):.4f}')
print(f'\nOriginal R² : {r2:.4f}')

---
## 8.  Cara Menyimpan Model

In [ ]:
os.makedirs('saved_models', exist_ok=True)
joblib.dump(ridge_cv_model, 'saved_models/ridge_productivity.pkl')
joblib.dump(list(X.columns), 'saved_models/feature_cols_prod_linreg.pkl')
print(' Ridge Productivity model tersimpan!')

---
## 9.  Cara Menggunakan Model Hasil Training

In [ ]:
loaded_model = joblib.load('saved_models/ridge_productivity.pkl')
loaded_cols = joblib.load('saved_models/feature_cols_prod_linreg.pkl')
print('Model dimuat ')

new_students = pd.DataFrame([
    # Mahasiswa aktif: banyak belajar, tidur cukup, mental health baik
    {'age': 20, 'gender': 1, 'academic_level': 1, 'study_hours': 6, 'self_study_hours': 3,
     'online_classes_hours': 2, 'social_media_hours': 1, 'gaming_hours': 0.5, 'sleep_hours': 7,
     'screen_time_hours': 4, 'exercise_minutes': 45, 'caffeine_intake_mg': 100,
     'part_time_job': 0, 'upcoming_deadline': 1, 'internet_quality': 2,
     'mental_health_score': 8, 'focus_index': 7, 'burnout_level': 2},
    # Mahasiswa pemalas: sedikit belajar, banyak main game, kurang tidur
    {'age': 22, 'gender': 0, 'academic_level': 2, 'study_hours': 1, 'self_study_hours': 0.5,
     'online_classes_hours': 0.5, 'social_media_hours': 5, 'gaming_hours': 4, 'sleep_hours': 5,
     'screen_time_hours': 10, 'exercise_minutes': 5, 'caffeine_intake_mg': 300,
     'part_time_job': 1, 'upcoming_deadline': 0, 'internet_quality': 1,
     'mental_health_score': 4, 'focus_index': 3, 'burnout_level': 7}
])[loaded_cols]

preds = loaded_model.predict(new_students)
print('\nPrediksi Productivity Score:')
for i, p in enumerate(preds):
    label = ' Sangat Produktif' if p >= 75 else (' Produktif' if p >= 50 else ' Kurang Produktif')
    print(f'  Mahasiswa {i+1}: Score = {p:.1f} → {label}')